# Stage 2 Notebook 61 - Exp2FFF DN-DETR + lambda_det=0 (lane-only effective)

**Isolate joint conflict from DAB-DETR instability.** NB59 hit val_lane_f1=0.623 at epoch 3 (project record) but the cls OSCILLATED: 0.458 ep1, 0.272 ep2, 0.623 ep3, 0.277 ep4, 0.594 ep11, 0.222 ep15, 0.240 ep20. Hypothesis: the oscillation is caused by det's gradient pushing the shared backbone, which moves the DAB anchor parameters between stable configurations.

Exp2FFF: set `lambda_det=0.0`, effectively training lane-only. Det loss still computed for logging but doesn't contribute to backward. If val_lane_f1 STABILIZES at 0.5+ without oscillation, joint conflict was the cause. If it still oscillates, DAB anchors are intrinsically unstable and we need a different stabilization strategy.

Same head as NB59 (DN-DETR, K=64, DAB anchors, DN denoising) and same long head_warmup (10 ep) + bb_mult=0.02. Single config change: `lambda_det: 1.5 -> 0.0`.

### Run mode
1. Smoke.
2. 20 epochs, limit=3000.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp56_rmt_gca_query64_dn_lane_only_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp56_rmt_gca_query64_dn_lane_only_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp56_rmt_gca_query64_dn_lane_only_joint_smoke.log
OK exp56_rmt_gca_query64_dn_lane_only_joint.yaml
  lane_shape=(1, 64, 72, 2) det_shape=(1, 4, 4)
  lane_loss=3.5176 det_loss=3.3441 grad_cos=-0.0894 lambda_lane=0.1163
  gate_stats={'gate/det_mean': 0.5014257431030273, 'gate/lane_mean': 0.49905550479888916, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp56_rmt_gca_query64_dn_lane_only_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'short20'
    EPOCHS = 20
    BATCH_SIZE = 8
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 50

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]
if LIMIT_TRAIN is not None:
    cmd.extend(['--limit-train', str(LIMIT_TRAIN)])

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('LIMIT_TRAIN:', LIMIT_TRAIN, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
LIMIT_TRAIN: 3000
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp56_rmt_gca_query64_dn_lane_only_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp56_rmt_gca_query64_dn_lane_only_joint_short20 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp56_rmt_gca_query64_dn_lane_only_joint_short20.tar --epochs 20 --batch-size 8 --limit-val 1000 --force-extract --print-every 50 --limit-train 3000
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp56_rmt_gca_query64_dn_lane_only_joint_short20.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp56_rmt_gca_query64_dn_lane_only_joint_short20_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp56_rmt_gca_query64_dn_lane_only_joint.yaml --curve-tar /co

0

## What to watch in Exp2FFF

Reference NB59: val_lane_f1 oscillates 0.22-0.62 across epochs, peak at ep3=0.623.

Pass criteria at epoch 20:
- **val_lane_f1 STAYS >= 0.45 from epoch 5 onwards** -- no oscillation = joint conflict was the cause.
- val_lane_best_f1 STAYS >= 0.50.
- gap stays >= 0.15.
- val_det may climb (we are not training it) but that's OK.

If val_lane_f1 still oscillates: DAB anchors are intrinsically unstable; need Exp2HHH-style aggressive LR/weight-decay throttling.